# 動画マスタの取得

チャンネルの全動画一覧（タイトル・投稿日時JST・動画長・Shorts判定）を取得してCSV保存する。
投稿数分析（`03_analyze/upload_count_analysis`）や日次ランキングのShorts判定の元データになる。

In [ ]:
#@title 🔧 セットアップ（パッケージinstall + Driveマウント）
GITHUB_OWNER = "asmrt"  # ← GitHubユーザー名に変更

from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q "git+https://{token}@github.com/{GITHUB_OWNER}/unofficial_sixfonia_analytics.git"
drive.mount("/content/drive")

In [ ]:
#@title ⚙️ 設定と取得
CHANNEL = "hima72"  #@param ["hima72", "sixfonia", "kosame", "illuma", "mikoto", "suchi", "lan"]

import pandas as pd
from sixfonia_analytics import auth, collect, config

youtube = auth.build_youtube()
rows = collect.fetch_video_master(youtube, config.channel_id_of(CHANNEL))

df = pd.DataFrame(rows)
df["published_at"] = pd.to_datetime(df["published_at"])
df = df.sort_values("published_at").reset_index(drop=True)
print(f"チャンネル: {CHANNEL} / 取得件数: {len(df)} / Shorts: {df['is_short'].sum()}件")
df.tail(10)

In [ ]:
#@title 💾 CSV保存（Drive）とダウンロード
DOWNLOAD = False  #@param {type:"boolean"}

path = collect.save_master_csv(rows, CHANNEL)

if DOWNLOAD:
    from google.colab import files
    files.download(str(path))